In [286]:
import pennylane as qml
from pennylane import numpy as np
import plotly.express as plot

In [ ]:
np.set_printoptions(formatter={'all': lambda x: f"{x:.3f}" if np.iscomplexobj(x) or np.isrealobj(x) else str(x)})

class MySearchAlgorithm:
    def __init__(self, nqbits : int = 4, shots : int | None = 500, analytic_mode = False, lookfor : list = [], use_simulator : bool = False):
        self.qbits = nqbits
        self.wires = range(0,nqbits)
        self.shots = None if analytic_mode else shots


        self.lookfor = [0 for _ in self.wires] if not lookfor else lookfor

        if use_simulator:
            from qiskit_ibm_runtime.fake_provider import FakeOslo as QCPU   ## CHOSE AN APPROPRIATE MODEL FOR THE NUMBER OF QBITS
            import qiskit_aer.noise as noise
            noise_model = noise.NoiseModel.from_backend(QCPU())
            self.dev = qml.device('qiskit.aer', wires=self.wires, noise_model=noise_model)
        else:
            self.dev = qml.device('default.qubit', wires=self.wires)

        self.data = {}

        def oracle(v = self.lookfor, w = self.wires):
            qml.Reflection(qml.BasisEmbedding(v, w), reflection_wires=w)

        def reflect(v = [], w = self.wires):
            if not v:
                v = [0 for _ in range(0,self.qbits)]
            qml.Reflection(qml.BasisEmbedding(v, w), reflection_wires=w)

        @qml.qnode(self.dev, shots=self.shots)
        def circuit(params = {}):
            if 'rep' not in params:
                params['rep'] = 0
            list(map(qml.H, self.wires))
            # repeat sqrt(N) - logN / 2 - 1 / 2 where N is the number of combinations
            for _ in range(0, pow(2, self.qbits / 2).__round__() - int((self.qbits) / 2) + params['rep']):
                params['oracle'](self.wires) if 'oracle' in params else oracle()
                list(map(qml.H, self.wires))
                reflect()
                list(map(qml.H, self.wires))
                

            return {
                'probs': qml.probs(),
                'shots': qml.sample() if self.shots else [],
                # 'state': qml.state() if not self.shots else [],
                'qbits-probs': { w : qml.probs([w]) for i, w in enumerate(self.wires)}
            }
    
        self.circuit = circuit

    def probs_to_bits(self, probs : dict, args : list=[]):
        from itertools import product
        for (arg, mode) in args:
            if arg not in probs or len(probs[arg]) == 0:
                continue
            if mode == 'str':
                probs[arg] = {str(''.join(map(str, id))): np.float64(probs[arg][i]) for i, id in enumerate(product([0, 1], repeat=int(np.log2(len(probs[arg])))))}
            if mode == 'int':
                probs[arg] = { i : np.complex64(probs[arg][i]) for i in range(0, len(probs[arg]))}
            if mode == 'bit':
                probs[arg] = { i : np.float64(probs[arg][i][1]) for i in range(0, len(probs[arg]))}
        return probs

    def get_circuit(self):
        return self.circuit

    def set_circuit(self, circuit):
        self.circuit = circuit
    
    def exec_circuit(self, *args, **kwargs):
        self.data = self.get_circuit()(*args, **kwargs)
        self.data = self.probs_to_bits(self.data, [('state', 'int'), ('probs', 'str'), ('qbits-probs', 'bit')])
        return self.data

    def draw_circuit(self, *args, **kwargs):
        print(qml.draw(self.get_circuit())(*args, **kwargs))

    def show_graph(self, key : str = 'probs', labels : dict = {"x": "Sequence", "y": "Occurrence"}, **kwargs):
        graph = plot.bar(x=self.data[key].keys(), y=self.data[key].values(), labels=labels, range_y=[0,1], height=500, width=800, text_auto=True, **kwargs)
        graph.show()

In [292]:
secret = [0,0,0]
def oracle(wires):
    qml.Reflection(qml.BasisEmbedding(secret + [1 for _ in range(0,len(wires) - len(secret))], wires), reflection_wires=wires) # Base oracle (uses 'secret' followed by ones)
    # qml.H(wires[-1])
    # qml.MultiControlledX(wires)
    # qml.H(wires[-1])

o = {'oracle': oracle}
s = 100

for n in [3, 4, 5]:
    alg = MySearchAlgorithm(shots=s, nqbits=n, use_simulator=True)
    alg.draw_circuit({'rep': 0} | o)

    alg.exec_circuit({'rep': 1} | o)
    alg.show_graph(title=f"{n} Qbits, 1 extra iterations")
    alg.show_graph('qbits-probs', title=f"{n} Qbits, 1 extra iterations")
    print('###########################################################################################')
    alg.exec_circuit({'rep': 0} | o)
    alg.show_graph(title=f"{n} Qbits, 0 extra iterations")
    alg.show_graph('qbits-probs', title=f"{n} Qbits, 0 extra iterations")
    print('###########################################################################################')
    alg.exec_circuit({'rep': -1} | o)
    alg.show_graph(title=f"{n} Qbits, -1 extra iterations")
    alg.show_graph('qbits-probs', title=f"{n} Qbits, -1 extra iterations")

0: ──H─╭Reflection(3.14,M0)──H─╭Reflection(3.14,M0)──H─╭Reflection(3.14,M0)──H ···
1: ──H─├Reflection(3.14,M0)──H─├Reflection(3.14,M0)──H─├Reflection(3.14,M0)──H ···
2: ──H─╰Reflection(3.14,M0)──H─╰Reflection(3.14,M0)──H─╰Reflection(3.14,M0)──H ···

0: ··· ─╭Reflection(3.14,M0)──H─┤  Probs  Sample  Probs
1: ··· ─├Reflection(3.14,M0)──H─┤  Probs  Sample  Probs
2: ··· ─╰Reflection(3.14,M0)──H─┤  Probs  Sample  Probs

M0 = 
[0.000 0.000 0.000]


###########################################################################################


###########################################################################################


0: ──H─╭Reflection(3.14,M0)──H─╭Reflection(3.14,M1)──H─╭Reflection(3.14,M0)──H ···
1: ──H─├Reflection(3.14,M0)──H─├Reflection(3.14,M1)──H─├Reflection(3.14,M0)──H ···
2: ──H─├Reflection(3.14,M0)──H─├Reflection(3.14,M1)──H─├Reflection(3.14,M0)──H ···
3: ──H─╰Reflection(3.14,M0)──H─╰Reflection(3.14,M1)──H─╰Reflection(3.14,M0)──H ···

0: ··· ─╭Reflection(3.14,M1)──H─┤  Probs  Sample  Probs
1: ··· ─├Reflection(3.14,M1)──H─┤  Probs  Sample  Probs
2: ··· ─├Reflection(3.14,M1)──H─┤  Probs  Sample  Probs
3: ··· ─╰Reflection(3.14,M1)──H─┤  Probs  Sample  Probs

M0 = 
[0.000 0.000 0.000 1.000]
M1 = 
[0.000 0.000 0.000 0.000]


###########################################################################################


###########################################################################################


0: ──H─╭Reflection(3.14,M0)──H─╭Reflection(3.14,M1)──H─╭Reflection(3.14,M0)──H ···
1: ──H─├Reflection(3.14,M0)──H─├Reflection(3.14,M1)──H─├Reflection(3.14,M0)──H ···
2: ──H─├Reflection(3.14,M0)──H─├Reflection(3.14,M1)──H─├Reflection(3.14,M0)──H ···
3: ──H─├Reflection(3.14,M0)──H─├Reflection(3.14,M1)──H─├Reflection(3.14,M0)──H ···
4: ──H─╰Reflection(3.14,M0)──H─╰Reflection(3.14,M1)──H─╰Reflection(3.14,M0)──H ···

0: ··· ─╭Reflection(3.14,M1)──H─╭Reflection(3.14,M0)──H─╭Reflection(3.14,M1)──H ···
1: ··· ─├Reflection(3.14,M1)──H─├Reflection(3.14,M0)──H─├Reflection(3.14,M1)──H ···
2: ··· ─├Reflection(3.14,M1)──H─├Reflection(3.14,M0)──H─├Reflection(3.14,M1)──H ···
3: ··· ─├Reflection(3.14,M1)──H─├Reflection(3.14,M0)──H─├Reflection(3.14,M1)──H ···
4: ··· ─╰Reflection(3.14,M1)──H─╰Reflection(3.14,M0)──H─╰Reflection(3.14,M1)──H ···

0: ··· ─╭Reflection(3.14,M0)──H─╭Reflection(3.14,M1)──H─┤  Probs  Sample  Probs
1: ··· ─├Reflection(3.14,M0)──H─├Reflection(3.14,M1)──H─┤  Probs  Sample  Probs
2: 

###########################################################################################


###########################################################################################
